# `create_clustering_app` — интерактивный разбор различий между кластерами

Bokeh-приложение для интерпретации 2D-эмбеддингов:

1. рисуем эмбеддинги объектов на плоскости;
2. инструментом **Lasso** выделяем точки и назначаем им кластер (имя + цвет);
3. выбираем два кластера, жмём «Сравнить» — обучается LightGBM (`boosting_type='rf'`) с таргетом
   «левый кластер vs правый», и строится SHAP beeswarm уже в пространстве **исходных признаков**.

Кластеры образуются в пространстве эмбеддингов, а их различия объясняются в исходных фичах.
История сравнений копится на странице (новое — сверху).

> beeswarm-график внутри приложения — самостоятельный тул, см. [`../beeswarm/`](../beeswarm/);
> здесь используется его встроенная копия (`get_beeswarm_fig`).

Поскольку приложение интерактивное, в выводе ячейки ничего не сохраняется — чтобы увидеть результат
без запуска, ниже скриншот работающего инструмента: слева кластеры, выделенные Lasso, снизу — beeswarm-сравнение
выбранной пары по исходным признакам.

![Интерактивный разбор кластеров](screenshots/app.png)

<details>
<summary>Продолжнение скриншота: история beeswarm-сравнений нескольких пар кластеров</summary>

![История сравнений](screenshots/comparison_history.png)

</details>

In [1]:
import os
os.environ["BOKEH_ALLOW_WS_ORIGIN"] = "*"

import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import io
import base64
from datetime import datetime

from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, Select, Button, TextInput, Div, ColorPicker
from bokeh.layouts import row, column
from bokeh.io import output_notebook

# Initialize Bokeh output in the notebook
output_notebook()

Loading BokehJS ...

In [2]:
# исходные данные и признаки (LightGBM для сравнения кластеров обучается внутри приложения)
df = pd.read_parquet('sber_data.parquet')
features = ['district_area', 'road_distance_1', 'road_distance_2',
       'year_of_construction', 'bulvar_ring_km', 'bus_station_distance',
       'cafe_count', 'fitness_center_distance', 'floor',
       'district_population', 'total_area', 'green_part',
       'green_zone_distance', 'healthcare_centers_count', 'id',
       'kitchen_area', 'kremlin_distance', 'leisure_count', 'living_area',
       'market_count', 'wall_material', 'floors_num', 'metro_minutes',
       'mkad_distance', 'rooms_num', 'office_count', 'park_distance',
       'product_type', 'public_transport_station_distance', 'sadovoe_km',
       'base_school_distance', 'sport_count', 'state', 'district_name',
       'malls_count', 'railway_station_distance', 'shadow_float_1',
       'shadow_float_2', 'shadow_float_3', 'shadow_float_4',
       'shadow_float_5', 'shadow_cat_big_1', 'shadow_cat_big_2',
       'shadow_cat_big_3', 'shadow_cat_big_4', 'shadow_cat_big_5',
       'shadow_cat_small_1', 'shadow_cat_small_2', 'shadow_cat_small_3',
       'shadow_cat_small_4', 'shadow_cat_small_5']

In [3]:
embeddings = np.load('sber_data_emb.npy')
embeddings.shape

(6264, 2)

In [4]:
def get_beeswarm_fig(shap_values, features_df, cat_feature_threshold=0.001, top_k=13, figsize=(10, 6), dots=1000):
    """
    Generates a matplotlib figure for an improved SHAP beeswarm plot.

    Parameters:
    -----------
    shap_values : np.ndarray
        Array of SHAP values calculated by the explainer.
    features_df : pd.DataFrame
        DataFrame containing the feature values used for the plot.
    cat_feature_threshold : float
        Minimum frequency required to explicitly show a specific category value.
    top_k : int
        Maximum number of features to display on the y-axis.
    figsize : tuple
        Dimensions of the generated matplotlib figure.
    dots : int
        Maximum number of points to sample and scatter per feature row.

    Returns:
    --------
    fig : matplotlib.figure.Figure
        The rendered matplotlib figure object.
    """
    # 1. Remove the expected_value column from LightGBM shap_values if present
    if shap_values.shape[1] == features_df.shape[1] + 1:
        shap_vals = shap_values[:, :-1]
    else:
        shap_vals = shap_values

    plot_data = []
    features = features_df.columns.tolist()

    # 2. Iterate through features to calculate SHAP statistics
    for i, col in enumerate(features):
        col_shap = shap_vals[:, i]
        col_vals = features_df[col].values

        is_categorical = pd.api.types.is_object_dtype(features_df[col]) or isinstance(features_df[col].dtype, pd.CategoricalDtype)

        if is_categorical:
            # 3. Process categorical features by splitting them into 'Feature = Value' pseudo-features
            value_counts = features_df[col].value_counts(normalize=True, dropna=True)
            for val, freq in value_counts.items():
                if freq >= cat_feature_threshold:
                    mask = (col_vals == val)
                    if mask.sum() > 0:
                        mean_abs_shap = np.mean(np.abs(col_shap[mask]))
                        plot_data.append({
                            'name': f"{col} = {val}",
                            'shap_array': col_shap[mask],
                            'val_array': np.ones(mask.sum()),
                            'is_nan_array': np.zeros(mask.sum(), dtype=bool),
                            'score': mean_abs_shap * freq,
                            'is_cat': True
                        })
        else:
            # 4. Process standard numerical features
            nan_mask = pd.isna(col_vals)
            plot_data.append({
                'name': col,
                'shap_array': col_shap,
                'val_array': col_vals,
                'is_nan_array': nan_mask,
                'score': np.mean(np.abs(col_shap)),
                'is_cat': False
            })

    # 5. Sort features by importance score and retain top_k
    plot_data.sort(key=lambda x: x['score'], reverse=True)
    plot_data = plot_data[:top_k]
    plot_data.reverse()

    # 6. Initialize figure and axes
    fig, ax = plt.subplots(figsize=figsize)
    ax.axvline(x=0, color='#999999', linestyle='-', linewidth=1.5, zorder=2)
    cmap = plt.get_cmap('coolwarm')
    y_labels = []

    # 7. Render scatter points for each feature
    for y_pos, feat in enumerate(plot_data):
        y_labels.append(feat['name'])
        ax.axhline(y=y_pos, color='#e0e0e0', linestyle='-', linewidth=0.8, zorder=1)

        full_shap = feat['shap_array']
        full_val = feat['val_array']
        full_nan = feat['is_nan_array']

        # 8. Downsample points to maintain rendering performance
        n_points = len(full_shap)
        idx = np.random.choice(n_points, size=min(n_points, dots), replace=False)
        shap_sample, val_sample, nan_sample = full_shap[idx], full_val[idx], full_nan[idx]
        valid_mask = ~nan_sample

        # 9. Plot valid points
        if valid_mask.sum() > 0:
            valid_shap, valid_vals = shap_sample[valid_mask], val_sample[valid_mask]
            jitter_valid = np.clip(np.random.normal(0, 0.08, size=valid_mask.sum()), -0.25, 0.25)

            if feat['is_cat']:
                colors = '#2ca02c'
            else:
                ranks = pd.Series(valid_vals).rank(pct=True).values
                colors = cmap(ranks)

            ax.scatter(valid_shap, y_pos + jitter_valid, color=colors, s=12, alpha=0.8, zorder=3, edgecolors='none')

        # 10. Plot missing values (NaN) distinctly
        if nan_sample.sum() > 0:
            jitter_nan = np.clip(np.random.normal(0, 0.015, size=nan_sample.sum()), -0.05, 0.05)
            ax.scatter(shap_sample[nan_sample], y_pos + jitter_nan, color='black', s=12, alpha=1.0, zorder=10, edgecolors='none')

    # 11. Finalize plot layout and colorbar
    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels)
    ax.set_ylabel("Features", fontsize=10)
    ax.set_xlabel("Contribution", fontsize=10)

    sm = ScalarMappable(cmap=cmap, norm=Normalize(vmin=0, vmax=1))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, aspect=40).set_ticks([0.0, 0.5, 1.0])
    plt.tight_layout()

    return fig


def create_clustering_app(embeddings, df, features, sample_size=5000):
    """
    Creates and launches an interactive Bokeh application for 2D embedding clustering
    and SHAP-based cluster interpretation.

    Parameters:
    -----------
    embeddings : np.ndarray
        2D array containing the coordinates of the embeddings.
    df : pd.DataFrame
        Original dataframe containing the raw feature data.
    features : list of str
        List of column names to be used as features for training the LightGBM model.
    sample_size : int
        Maximum number of data points to display and train on to ensure UI responsiveness.
    """

    # 1. Downsample the data to prevent UI freezing
    if len(embeddings) > sample_size:
        idx = np.random.choice(len(embeddings), size=sample_size, replace=False)
        emb_sample = embeddings[idx]
        df_sample = df.iloc[idx].reset_index(drop=True)
    else:
        emb_sample = embeddings
        df_sample = df.copy().reset_index(drop=True)

    df_features = df_sample[features].copy()

    # 2. Convert object columns to category dtype for LightGBM compatibility
    cat_cols = df_features.select_dtypes(include=['object']).columns
    for c in cat_cols:
        df_features[c] = df_features[c].astype('category')

    def app(doc):
        # 3. Create Bokeh data source
        source = ColumnDataSource(data=dict(
            x=emb_sample[:, 0],
            y=emb_sample[:, 1],
            color=['#d3d3d3'] * len(emb_sample),
            cluster_name=['N/A'] * len(emb_sample)
        ))

        # Track history of comparisons to handle UI states
        compared_pairs = set()

        # --- UI COMPONENTS: Plot ---
        p = figure(title="2D Embeddings (Use Lasso tool to select points)",
                   tools="lasso_select,pan,wheel_zoom,reset",
                   width=600, height=500)
        p.scatter('x', 'y', color='color', source=source, size=6, alpha=0.7)

        # --- UI COMPONENTS: Cluster Assignment ---
        color_picker = ColorPicker(color="#1f77b4", title="Cluster Color:")
        name_input = TextInput(value="cluster_1", title="Cluster Name:")
        assign_btn = Button(label="Assign Cluster", button_type="success")
        cluster_list_div = Div(text="<b>Created Clusters:</b><br>N/A")

        # --- UI COMPONENTS: Comparison & ML ---
        left_select = Select(title="Left Cluster (Class 0):", value="N/A", options=["N/A"])
        right_select = Select(title="Right Cluster (Class 1):", value="N/A", options=["N/A"])
        top_k_input = TextInput(title="top_k features:", value="15")
        cat_thresh_input = TextInput(title="cat_feature_threshold:", value="0.005")

        status_div = Div(text="")
        compare_btn = Button(label="Сравнить кластеры", button_type="primary")

        # Container for the history of generated SHAP plots
        plot_history_title = Div(text="<h3>История SHAP сравнений, новое сверху</h3>")
        plot_history_layout = column()

        # --- LOGIC CALLBACKS ---

        def update_ui_status(attr, old, new):
            """Updates the button text and warning message based on current selection."""
            left = left_select.value
            right = right_select.value

            if left == right:
                status_div.text = ""
                compare_btn.label = "Сравнить кластеры"
                return

            pair = tuple(sorted([left, right]))
            if pair in compared_pairs:
                status_div.text = "<i style='color:#d9534f; font-size: 0.9em;'>Эта пара кластеров ранее сравнивалась. Если хотите пересчитать и добавить новый график сверху - нажмите 'обновить SHAP-график'</i>"
                compare_btn.label = "Обновить SHAP-график"
            else:
                status_div.text = ""
                compare_btn.label = "Сравнить кластеры"

        left_select.on_change('value', update_ui_status)
        right_select.on_change('value', update_ui_status)

        def assign_cluster_callback():
            """Assigns selected color and name to the points currently captured by Lasso."""
            selected_idx = source.selected.indices
            if not selected_idx:
                return

            c_name = name_input.value
            c_color = color_picker.color

            # Use dtype=object to prevent numpy string truncation (the 3-letter bug fix)
            new_colors = np.array(source.data['color'], dtype=object)
            new_names = np.array(source.data['cluster_name'], dtype=object)

            new_colors[selected_idx] = c_color
            new_names[selected_idx] = c_name

            source.data['color'] = new_colors.tolist()
            source.data['cluster_name'] = new_names.tolist()
            source.selected.indices = []

            unique_clusters = sorted(list(set(new_names)))
            left_select.options = unique_clusters
            right_select.options = unique_clusters
            cluster_list_div.text = "<b>Created Clusters:</b><br>" + "<br>".join(unique_clusters)
            update_ui_status(None, None, None)

        def compare_clusters_callback():
            """Trains the ML model on selected clusters and prepends the new SHAP plot to history."""
            left = left_select.value
            right = right_select.value

            if left == right:
                status_div.text = "<b style='color:red;'>Ошибка: Выберите два разных кластера!</b>"
                return

            status_div.text = "<b>Обучение модели, подождите...</b>"

            # Vectorized filtering for target classes
            names = np.array(source.data['cluster_name'])
            mask = (names == left) | (names == right)

            X_train = df_features[mask]
            y_train = (names[mask] == right).astype(int)

            if len(y_train) < 10:
                status_div.text = "<b style='color:red;'>Слишком мало точек для обучения. Выделите больше!</b>"
                return

            # Train LightGBM Random Forest implementation
            clf = lgb.LGBMClassifier(
                boosting_type='rf',
                objective='binary',
                n_estimators=40,
                bagging_freq=1,
                bagging_fraction=0.8,
                feature_fraction=0.8,
                random_state=42,
                n_jobs=-1,
                verbose=-1
            )

            clf.fit(X_train, y_train)
            shap_values = clf.predict(X_train, pred_contrib=True)

            # Extract UI configuration parameters
            t_k = int(top_k_input.value)
            c_thresh = float(cat_thresh_input.value)

            # Generate the visualization
            fig = get_beeswarm_fig(
                shap_values=shap_values,
                features_df=X_train,
                cat_feature_threshold=c_thresh,
                top_k=t_k
            )

            # Convert Matplotlib to Base64 to bypass Bokeh rendering limitations
            buf = io.BytesIO()
            fig.savefig(buf, format='png', bbox_inches='tight')
            buf.seek(0)
            img_str = base64.b64encode(buf.read()).decode('utf-8')
            plt.close(fig)

            # Mark this pair as evaluated
            pair = tuple(sorted([left, right]))
            compared_pairs.add(pair)

            # Generate the contextual header for the plot
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            plot_header = f"<b>Сравнение {left} и {right}</b>. Время {timestamp}, top_k {t_k}, threshold {c_thresh}"

            # Prepend the newly generated plot to the history layout
            new_plot_item = Div(text=f"{plot_header}<br><img src='data:image/png;base64,{img_str}' style='margin-bottom: 20px;'>", width=800)
            plot_history_layout.children = [new_plot_item] + list(plot_history_layout.children)

            update_ui_status(None, None, None)

        assign_btn.on_click(assign_cluster_callback)
        compare_btn.on_click(compare_clusters_callback)

        # --- LAYOUT ASSEMBLY ---
        clustering_controls = column(name_input, color_picker, assign_btn, cluster_list_div, width=250)
        compare_controls = column(left_select, right_select, top_k_input, cat_thresh_input, status_div, compare_btn, width=300)

        top_row = row(p, clustering_controls, compare_controls)
        history_section = column(plot_history_title, plot_history_layout)

        layout = column(top_row, history_section)

        doc.add_root(layout)

    # Note: Keep os.environ["BOKEH_ALLOW_WS_ORIGIN"] set before this cell if running in VS Code
    show(app)

### Запуск

In [5]:
create_clustering_app(
    embeddings=embeddings,
    df=df,
    features=features,
    sample_size=5000
)